# Assignment 3

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model](https://open.spotify.com/track/4R3AU2pjv8ge2siX1fVbZs?si=b2712f32da0e4358) or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [78]:
# Load the libraries as required.
import pandas as pd

In [79]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   FFMC     517 non-null    float64
 5   DMC      517 non-null    float64
 6   DC       517 non-null    float64
 7   ISI      517 non-null    float64
 8   temp     517 non-null    float64
 9   RH       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [87]:
X = fires_dt.drop(columns=['area'])
Y = fires_dt['area']

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

print(X_train.head())

print(Y_train.head())


     coord_x  coord_y month  day  FFMC    DMC     DC   ISI  temp    RH  wind  \
311      6.0      3.0   sep  sun  92.4  105.8  758.1   9.9  24.8  28.0   1.8   
368      6.0      5.0   sep  sat  91.2   94.3  744.4   8.4  16.8  47.0   4.9   
23       7.0      4.0   aug  sat  90.2  110.9  537.4   6.2  19.5  43.0   5.8   
271      8.0      6.0   aug  tue  92.1  152.6  658.2  14.3  20.1  58.0   4.5   
299      6.0      5.0   jun  sat  53.4   71.0  233.8   0.4  10.6  90.0   2.7   

     rain  
311   0.0  
368   0.0  
23    0.0  
271   0.0  
299   0.0  
311    14.29
368    12.64
23      0.00
271     9.27
299     0.00
Name: area, dtype: float64


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [88]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


numerical_cols = ['coord_x', 'coord_y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
categorical_cols = ['month', 'day']

# check for NaN Values
nan_summary = fires_dt.isnull().sum()
print("NaN Summary Before Imputation:\n", nan_summary)

# impute Missing Values
imputer = SimpleImputer(strategy='mean')  # Choose appropriate strategy
fires_dt[numerical_cols] = imputer.fit_transform(fires_dt[numerical_cols])

# print the imputation outcome
nan_summary_after = fires_dt.isnull().sum()
print("NaN Summary After Imputation:\n", nan_summary_after)



NaN Summary Before Imputation:
 coord_x    0
coord_y    0
month      0
day        0
FFMC       0
DMC        0
DC         0
ISI        0
temp       0
RH         0
wind       0
rain       0
area       0
dtype: int64
NaN Summary After Imputation:
 coord_x    0
coord_y    0
month      0
day        0
FFMC       0
DMC        0
DC         0
ISI        0
temp       0
RH         0
wind       0
rain       0
area       0
dtype: int64


In [89]:
# use debug function
# from sklearn.base import BaseEstimator, TransformerMixin

class DebugTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, name):
        self.name = name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        print(f"NaN Summary After {self.name}:\n", pd.DataFrame(X).isnull().sum())
        return X

preproc = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('impute', SimpleImputer(strategy='mean')),
            ('debug_impute', DebugTransformer('Imputation')),
            ('scale', StandardScaler()),
            ('debug_scale', DebugTransformer('Scaling'))
        ]), numerical_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ]
)


In [90]:
# check for my train set null values and data type
print(Y_train.isnull().sum())
print(Y_train.dtypes)


0
float64


In [91]:
#check for correct type of data for the columns (clean data)
# print(X_train.dtypes)


In [92]:
# try to simplify my model to use a gridSearchCV function on
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV


model = Pipeline(steps=[
    ('preprocessor', preproc),
    ('regressor', LinearRegression())
])

grid_search = GridSearchCV(estimator=model, param_grid={}, cv=5, scoring='neg_mean_squared_error', verbose=2, n_jobs=-1)
grid_search.fit(X_train, Y_train)

# Print the best score -- to check for the error, as it is the error message returned for evaluation
print("Best Score:", grid_search.best_score_)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
NaN Summary After Imputation:
 0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64
NaN Summary After Scaling:
 0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64
Best Score: nan


c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan]
  warnings.warn(


In [93]:
# only check the train dataset, to address the error message
X_train_subset = X_train.sample(frac=0.1, random_state=42)
y_train_subset =Y_train.loc[X_train_subset.index]

grid_search.fit(X_train_subset, y_train_subset)
print("Best Score on Subset:", grid_search.best_score_)


Fitting 5 folds for each of 1 candidates, totalling 5 fits
NaN Summary After Imputation:
 0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64
NaN Summary After Scaling:
 0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
dtype: int64
Best Score on Subset: nan


c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan]
  warnings.warn(


In [94]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, FunctionTransformer
import numpy as np

numerical_cols = ['coord_x', 'coord_y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
categorical_cols = ['month', 'day']

#preproc1 pipeline using standard scaler for the numerical variables and the one-hot encoding for the categorical variables
preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ]
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [95]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numerical_cols = ['coord_x', 'coord_y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
categorical_cols = ['month', 'day']

#preproc2 pipeline using minmax to scale the numeric variables, and the log function for the non-linear transformation; one-hot encoding of catergorical variables
preproc2 = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('impute', SimpleImputer(strategy='mean')),  # Choose appropriate strategy
            ('scale', StandardScaler())
        ]), numerical_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ]
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [96]:
# Pipeline A = preproc1 + baseline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, BaggingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score

knn_model = KNeighborsRegressor()
ridge_model = Ridge()
bagging_model = BaggingRegressor()
boosting_model = GradientBoostingRegressor()

Pipeline_A = Pipeline(steps=[('preprocessing', preproc1), ('regressor', knn_model)])

In [97]:
# Pipeline B = preproc2 + baseline
Pipeline_B = Pipeline(steps=[('preprocessing', preproc2), ('regressor', ridge_model)])

In [98]:
# Pipeline C = preproc1 + advanced model
Pipeline_C = Pipeline(steps=[('preprocessing', preproc1), ('regressor', bagging_model)])


In [99]:
# Pipeline D = preproc2 + advanced model
Pipeline_D = Pipeline(steps=[('preprocessor', preproc2), ('regressor', boosting_model)])
    

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [100]:
grid_search_A = GridSearchCV(Pipeline_A, {'regressor__n_neighbors': [3, 5, 7, 9]},
                             cv=5, scoring='neg_mean_squared_error')

grid_search_B = GridSearchCV(Pipeline_B, {'regressor__alpha': [0.1, 1.0, 10.0, 100.0]},
                             cv=5, scoring='neg_mean_squared_error')

grid_search_C = GridSearchCV(Pipeline_C, {'regressor__n_estimators': [10, 50, 100, 200]},
                              cv=5, scoring='neg_mean_squared_error')

grid_search_D = GridSearchCV(Pipeline_D, {'regressor__n_estimators': [50, 100, 200, 300], 'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2]},
                             cv=5, scoring='neg_mean_squared_error')

In [101]:
grid_search_A.fit(X_train, Y_train)
grid_search_B.fit(X_train, Y_train)
grid_search_C.fit(X_train, Y_train)
grid_search_D.fit(X_train, Y_train)

c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\model_selection\_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\model_selection\_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\metrics\_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\metrics\_scorer.py", line 371, in _score
    y_pred = method_caller(
  File "c:\Users\Anca\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\metrics\_scorer.py", line 89, in _cached_call
    result, _ = _get_response_values(
  File "c:\Users\Anca

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('impute',
                                                                                          SimpleImputer()),
                                                                                         ('scale',
                                                                                          StandardScaler())]),
                                                                         ['coord_x',
                                                                          'coord_y',
                                                                          'FFMC',
                                                                          'DMC',
                                                                          'DC',
                                                                          'ISI',
                                                                          'temp',
                                                                          'RH',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('cat',
                                                                         OneHotEncoder(),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor',
                                        GradientBoostingRegressor())]),
             param_grid={'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
                         'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='neg_mean_squared_error')

# Evaluate

+ Which model has the best performance?

In [102]:
best_params_A = grid_search_A.best_params_
best_params_B = grid_search_B.best_params_
best_params_C = grid_search_C.best_params_
best_params_D = grid_search_D.best_params_

scores_A = grid_search_A.best_score_
scores_B = grid_search_B.best_score_
scores_C = grid_search_C.best_score_
scores_D = grid_search_D.best_score_

print("Pipeline A Best Parameters:", best_params_A)
print("Pipeline A Best Score:", scores_A)
print("Pipeline B Best Parameters:", best_params_B)
print("Pipeline B Best Score:", scores_B)
print("Pipeline C Best Parameters:", best_params_C)
print("Pipeline C Best Score:", scores_C)
print("Pipeline D Best Parameters:", best_params_D)
print("Pipeline D Best Score:", scores_D)

Pipeline A Best Parameters: {'regressor__n_neighbors': 3}
Pipeline A Best Score: nan
Pipeline B Best Parameters: {'regressor__alpha': 0.1}
Pipeline B Best Score: nan
Pipeline C Best Parameters: {'regressor__n_estimators': 10}
Pipeline C Best Score: nan
Pipeline D Best Parameters: {'regressor__learning_rate': 0.01, 'regressor__n_estimators': 50}
Pipeline D Best Score: nan


# Export

+ Save the best performing model to a pickle file.

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_3_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-3`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_3.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.